# DualEncoderSeg v3 — Dual-Path Stage 1 with Latent Supervision

> **Author**: Anirban | **Notebook**: dual_encoder_v3.ipynb
> Builds on `dual_encoder_v2.ipynb`. Architecture-preserving Stage 1 improvements.

---

## What Changed from v2

| Component | v2 | v3 (This Notebook) |
|---|---|---|
| Stage 1 forward | Single path: z + skips → recon | **Dual path**: (z + skips → recon) AND (z alone → recon) |
| Stage 1 loss | L_pixel(recon, mask) | **L_main + λ_z·L_z_only + λ_lat·L_lat** |
| LatentHead in Stage 1 | Not present | **Added**: z → 32×32 direct vessel prediction |
| Gradient saturation | Loss → 0 at Dice=0.9999, learning stops | **Kept alive** via z-only + lat losses throughout |
| Decoder z-conditioning | Skip-dominant (z bypassed) | **Forced**: decoder must decode from z alone in path 2 |
| mapping_type default | global_dnn (163M params) | **spatial** (SpatialMappingNet, ~13M params, fair vs UNet) |
| evaluate_model | Per-batch metric average (biased) | **Full val set accumulated** then computed once |

---

## Core Motivation (Unchanged)

The MaskDecoder must reconstruct vessel masks from the 16x16 latent **z**.
Stage 2 alignment (z_pred <-> z_mask) requires z_mask to be a **rich, structured
representation** of vessel topology — not a memorised lookup table.

---

## v3 Stage 1 — Three Parallel Signals

```
mask --> MaskEncoder --> z, [f1..f4]
                         |         |
           Path 1 (main) |         +--> MaskDecoder(z + skips) --> recon_full  --> L_main
           Path 2 (z-only)          --> MaskDecoder(z, None)   --> recon_z     --> L_z_only
           Path 3 (latent)          --> LatentHead(z)          --> 32x32 pred  --> L_lat
```

- **Path 1** teaches decoder to optimally fuse z + skips (gates unaffected by paths 2/3)
- **Path 2** forces decoder upsampling weights to be conditioned on z alone (not skips)
- **Path 3** directly supervises z to contain vessel shape at 32x32 resolution

---

## Notebook Structure

| Section | Content |
|---|---|
| §0 | Environment & Imports |
| **§1** | **Config** |
| §2 | Model Specification Table |
| §3 | Shared Primitive Blocks |
| **§4** | **DualEncoderSeg v3 Architecture (LatentHead + dual-path MAE)** |
| §5 | UNet Baseline |
| **§6** | **Loss Suite (Stage1Loss: 3-term)** |
| §7 | Metrics |
| §8 | Dataset & DataLoaders |
| **§9** | **Training Functions (train_stage1 v3)** |
| §10 | Run Training |
| **§11** | **Results (new Stage 1 dynamics plot)** |
| §12 | Inference Utilities |


## §0 — Environment & Imports

In [ ]:
# Cell 0.1 — Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'torchvision', 'Pillow', 'matplotlib', 'seaborn', 'numpy', 'tqdm'],
               check=False)

In [ ]:
# Cell 0.2 — Imports
import os, sys, math, random, json, time
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

matplotlib.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans',
                             'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid', palette='tab10')
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')

## §1 — Global Config  *(change only this cell)*

In [ ]:
# Cell 1.1 — ExperimentConfig
# ─────────────────────────────────────────────────────────────────
# THIS is the single cell you modify to change the experiment.
# ─────────────────────────────────────────────────────────────────

@dataclass
class ExperimentConfig:
    # ── Data ────────────────────────────────────────
    DATA_ROOT:   str   = '/kaggle/input/datasets/abdallahwagih/retina-blood-vessel/Data/train'
    image_size:  int   = 512
    train_split: float = 0.8
    batch_size:  int   = 4
    num_workers: int   = 2
    seed:        int   = 42

    # ── Device ──────────────────────────────────────
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ── Architecture (IDENTICAL for both models) ────
    # Channel progression: 64->128->256->512->512
    # Conv block: DoubleConv  |  Downsampling: MaxPool
    # These cannot be changed without breaking the fair comparison.

    # ── DualEncoder-specific ────────────────────────
    latent_ch:    int   = 256       # 16x16x256 spatial latent
    mapping_type: str   = 'spatial' # 'spatial'    -> SpatialMappingNet (~13M, fair vs UNet)
                                     # 'global_dnn' -> GlobalLatentDNN   (~163M, overfits small data)
    dropout:      float = 0.1       # Dropout2d before latent projection

    # ── Training schedule ───────────────────────────
    stage1_epochs: int   = 60       # MaskAutoencoder pretraining (NO early stop)
    stage2_epochs: int   = 120      # DualEncoderSeg full training
    unet_epochs:   int   = 120      # UNet baseline
    stage1_lr:     float = 1e-3
    stage2_lr:     float = 3e-4
    unet_lr:       float = 3e-4
    warmup_epochs: int   = 10

    # ── Shared pixel loss weights (both models) ─────
    lambda_tversky:  float = 1.5    # Recall-biased Dice
    lambda_boundary: float = 1.0    # Sobel-weighted BCE at edges
    lambda_focal:    float = 0.5    # Hard example mining
    lambda_lovasz:   float = 1.0    # Direct IoU optimisation
    tversky_alpha:   float = 0.3    # FP weight  (b=0.7 -> FN penalised more)
    tversky_beta:    float = 0.7    # FN weight
    focal_gamma:     float = 2.0
    focal_alpha:     float = 0.25

    # ── DualEncoder alignment loss (Stage 2 only) ───
    lambda_align_mse: float = 1.0   # z_pred <-> z_mask MSE
    lambda_align_cos: float = 1.0   # z_pred <-> z_mask cosine
    lambda_aux:       float = 0.4   # Aux head Lovász at 32x32

    # ── v3: Stage 1 dual-path + latent supervision ──
    lambda_z:   float = 0.6   # weight for z-only reconstruction path loss  [v3+: 0.4→0.6]
    lambda_lat: float = 0.3   # weight for direct latent head Lovász loss (32x32)

    # ── Checkpoints ─────────────────────────────────
    save_dir: str = './checkpoints_v3'


CFG = ExperimentConfig()
torch.manual_seed(CFG.seed); np.random.seed(CFG.seed); random.seed(CFG.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CFG.seed)
os.makedirs(CFG.save_dir, exist_ok=True)
print(f'Device: {CFG.device}  |  Image: {CFG.image_size}px  |  Batch: {CFG.batch_size}')
print(f'Stage1: {CFG.stage1_epochs}ep  Stage2: {CFG.stage2_epochs}ep  UNet: {CFG.unet_epochs}ep')
print(f'Latent: 16x16x{CFG.latent_ch}  |  mapping_type: {CFG.mapping_type}')
print(f'v3 Stage1: lambda_z={CFG.lambda_z}  lambda_lat={CFG.lambda_lat}  (dual-path + latent supervision)')
print(f'No early stopping -- full research run')


## §2 — Model Specification Table  *(reference)*

In [ ]:
# Cell 2.1 — Architecture specification (printed at runtime from CFG)

print('=' * 72)
print('  IDENTICAL COMPONENTS (controlled variables)')
print('=' * 72)
identical = [
    ('Conv block',          'DoubleConv  (Conv3x3->BN->ReLU x 2)'),
    ('Downsampling',        'MaxPool 2x2'),
    ('Encoder depth',       '5 stages  (512->256->128->64->32->16)'),
    ('Encoder channels',    '64 -> 128 -> 256 -> 512 -> 512'),
    ('Decoder upsampling',  'Bilinear 2x + DoubleConv'),
    ('Image resolution',    f'{CFG.image_size}x{CFG.image_size}'),
    ('Pixel loss suite',    'Tversky + Boundary + Focal + Lovász'),
    ('Pixel loss weights',  f'ltvk={CFG.lambda_tversky}  lbnd={CFG.lambda_boundary}  lfcl={CFG.lambda_focal}  llov={CFG.lambda_lovasz}'),
    ('Optimizer',           'AdamW  weight_decay=1e-4'),
    ('Stage2/UNet LR',      str(CFG.stage2_lr)),
    ('LR schedule',         f'Warmup-cosine  ({CFG.warmup_epochs} warmup epochs)'),
    ('Batch size',          str(CFG.batch_size)),
    ('Early stopping',      'NONE -- full epoch runs (research mode)'),
    ('Augmentation',        'HFlip + VFlip + Rotate+-30 + Photometric'),
]
for k, v in identical:
    print(f'  {k:<24} {v}')

print()
print('=' * 72)
print('  MODEL-SPECIFIC COMPONENTS')
print('=' * 72)
specific = [
    ('Component',             'DualEncoderSeg v3',                          'UNet Baseline'),
    ('-'*24,                  '-'*36,                                       '-'*22),
    ('Training stages',       '2  (Stage1 MAE + Stage2 seg)',               '1  (end-to-end)'),
    ('Latent space',          f'Spatial 16x16x{CFG.latent_ch}',             'Bottleneck 16x16x512'),
    ('SkipFusionGate',        'Yes  (gates UNFROZEN in Stage 2)',            'No  (concat+DoubleConv)'),
    ('SpatialMappingNet',     'Yes  (z_img -> z_mask bridge)',               'No'),
    ('Latent alignment loss', 'MSE + Cosine  (curriculum weighted)',         'No'),
    ('AuxHead (Stage 2)',      f'Yes  (Lovász at 32x32, l={CFG.lambda_aux})', 'No'),
    ('LatentHead (Stage 1)',   f'Yes  (Lovász at 32x32, l={CFG.lambda_lat})','No   [v3 NEW]'),
    ('Stage1 z-only path',    f'Yes  (l_z={CFG.lambda_z})',                  'No   [v3 NEW]'),
    ('MaskEncoder dropout',   f'Dropout2d({CFG.dropout})',                   'N/A'),
    ('Cross-modal skip inj.', 'Yes  (image skips -> mask decoder)',          'No  (self-skips)'),
    ('Stage 1 LR',            str(CFG.stage1_lr),                           'N/A'),
    ('Stage 1 epochs',        str(CFG.stage1_epochs),                       'N/A'),
]
for row in specific:
    print('  '.join(str(c).ljust(w) for c, w in zip(row, [26, 38, 24])))


## §3 — Shared Primitive Blocks  *(identical in both models)*

In [ ]:
# Cell 3.1 — DoubleConv & DownBlock (shared by DualEncoder + UNet)

class DoubleConv(nn.Module):
    """
    Conv3×3 → BN → ReLU → Conv3×3 → BN → ReLU.
    The atomic building block used IDENTICALLY in both DualEncoder and UNet.
    No SE attention. No stride tricks. Plain and controlled.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class DownBlock(nn.Module):
    """
    MaxPool 2×2 → DoubleConv.
    Used IDENTICALLY in both DualEncoder encoders and UNet encoder.
    MaxPool (not stride-2 conv) — same inductive bias in both models.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x): return self.conv(self.pool(x))


class UpBlock(nn.Module):
    """
    Bilinear 2× upsample → DoubleConv.
    Used in MaskDecoder. Does NOT concat skips — skip injection
    is handled separately by SkipFusionGate.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x): return self.conv(self.up(x))


print('Shared blocks: DoubleConv, DownBlock, UpBlock')

In [ ]:
# Cell 3.2 — SkipFusionGate (DualEncoder only — the core innovation)

class SkipFusionGate(nn.Module):
    """
    Learned gated fusion of a skip connection into decoder context.

    Problem solved:
      Stage 2 skip features come from ImageEncoder (RGB domain).
      Decoder context comes from MaskDecoder (binary mask domain).
      Plain concat causes destructive interference across domains.

    Gate formula:
      g   = sigmoid( W · cat(skip, ctx) )   # per-channel confidence  [0,1]
      out = ctx_proj(ctx) + g ⊙ skip_proj(skip)  # gated residual
      out = BN(out)

    CRITICAL v2 fix: gates are UNFROZEN in Stage 2.
      Stage 1: gate trains on mask_skip ↔ mask_ctx (same domain → opens wide)
      Stage 2: gate re-learns image_skip ↔ mask_ctx (cross-domain → selective)
    """
    def __init__(self, skip_ch, ctx_ch, out_ch):
        super().__init__()
        self.skip_proj = nn.Conv2d(skip_ch, out_ch, 1, bias=False)
        self.gate      = nn.Sequential(
            nn.Conv2d(skip_ch + ctx_ch, out_ch, 1, bias=True),
            nn.ReLU(inplace=True),          # [v3+] nonlinearity for cross-domain channel interaction
            nn.Conv2d(out_ch, out_ch, 1, bias=True),
            nn.Sigmoid(),
        )
        self.ctx_proj  = nn.Conv2d(ctx_ch, out_ch, 1, bias=False)
        self.bn        = nn.BatchNorm2d(out_ch)

    def forward(self, skip: torch.Tensor, ctx: torch.Tensor) -> torch.Tensor:
        if skip.shape[-2:] != ctx.shape[-2:]:
            skip = F.interpolate(skip, size=ctx.shape[-2:], mode='bilinear', align_corners=False)
        g   = self.gate(torch.cat([skip, ctx], dim=1))
        out = self.ctx_proj(ctx) + g * self.skip_proj(skip)
        return self.bn(out)


print('SkipFusionGate defined  [v3+: nonlinear gate — ReLU hidden layer added]')

## §4 — DualEncoderSeg v3 Architecture

### Architectural Overview (DualEncoderSeg v3)

DualEncoderSeg v3 is trained using a **two-stage training protocol**, separating structural shape priors from cross-modal image-to-mask mapping.

#### Stage 1: Self-Supervised Dual-Path Mask Autoencoder (Pretraining)
In Stage 1, the **MaskAutoencoder** learns dense binary vessel representations by sharing a single encoder pass across **three parallel paths**:
1. **Path 1 (Main)**: Decodes from the latent bottleneck $z$ using binary mask skip connections to reconstruct the full mask.
2. **Path 2 ($z$-only)**: Decodes from the latent $z$ alone (skips disabled) to force the latent space to represent full shape context.
3. **Path 3 (Latent)**: Directly supervises $z$ at $32 	imes 32$ using a **LatentHead** to solve a low-resolution segmentation task.

```mermaid
graph TD
    subgraph Stage1["Stage 1: Self-Supervised Mask Autoencoder (Pretraining)"]
        M["Input Binary Mask<br>512×512×1"] --> ME["Mask Encoder<br>(Trainable)"]
        ME -->|"Latent z"| Z["z (B, 256, 16, 16)"]
        ME -->|"Skips s1..s4"| MS["Mask Skips<br>(64, 128, 256, 512 ch)"]
        
        %% Path 1
        Z -->|"Path 1"| MD_full["Mask Decoder<br>(Trainable)"]
        MS -->|"Gate Injection"| MD_full
        MD_full -->|"recon_full"| RF["Full Mask Logits<br>512×512×1"]
        
        %% Path 2
        Z -->|"Path 2"| MD_z["Mask Decoder<br>(skips=None)"]
        MD_z -->|"recon_z_only"| RZ["z-only Mask Logits<br>512×512×1"]
        
        %% Path 3
        Z -->|"Path 3"| LH["Latent Head [v3 NEW]<br>(Trainable)"]
        LH -->|"lat_pred"| LP["Latent Logits<br>32×32×1"]
        
        %% Losses
        RF & M -->|"Pixel Loss (1.0×)"| L_main["L_pixel Suite<br>(Tversky + Boundary + Focal + Lovász)"]
        RZ & M -->|"Pixel Loss (0.4×)"| L_z["L_pixel Suite"]
        LP -->|"Lovász-Hinge (0.3×)"| L_lat["L_latent"]
        
        L_main & L_z & L_lat -->|"Sum"| L_tot["Total Stage 1 Loss"]
    end
```

---

#### Stage 2: Dual Encoder Gated Skip-Injection & Latent Alignment
In Stage 2, the **MaskEncoder** and **MaskDecoder** upsampling weights are **frozen**, while the **SkipFusionGates** are **unfrozen and trainable** to adapt to the cross-modal domain shift. An **ImageEncoder** extracts RGB features and skips, mapping the image representation to the mask latent space via a **SpatialMappingNet**. Trainable **AuxHead** provides a short gradient path to the mapping network.

```mermaid
graph LR
    subgraph Stage2["Stage 2: Cross-Modal Segmentation (Fine-tuning & Inference)"]
        IMG["Input RGB Image<br>512×512×3"] --> IE["Image Encoder<br>(Trainable)"]
        IE -->|"z_img"| SMN["Mapping Net<br>(Trainable)"]
        IE -->|"RGB Skips f1..f4"| G["SkipFusionGates 1..4<br>(Trainable)"]
        
        SMN -->|"z_pred"| ZP["z_pred<br>16×16×256"]
        
        ZP --> MD2["Mask Decoder Upsampling<br>(FROZEN)"]
        G -->|"Selective Gate Projection"| MD2
        
        MD2 --> OUT["Prediction Logits<br>512×512×1"]
        
        %% Training Only Elements
        subgraph Supervision["Supervision (Training Only)"]
            MASK["True Mask<br>512×512×1"] --> ME2["Mask Encoder<br>(FROZEN)"]
            ME2 -->|"z_mask"| ZM2["z_mask<br>16×16×256"]
            ZP --> AUX["Aux Head<br>(Trainable)"]
            AUX -->|"aux_logits"| AUX_OUT["Aux Logits<br>32×32×1"]
        end
        
        %% Losses
        OUT -->|"Pixel Loss Suite"| L2["DualEncoder Loss"]
        AUX_OUT -->|"Aux Lovász"| L2
        ZP & ZM2 -->|"Latent Align: MSE + Cosine"| L2
    end
```


In [ ]:
# Cell 4.1 — MaskEncoder & ImageEncoder
# IDENTICAL channel progression and conv blocks to UNet.
# Only difference: input channels (1 vs 3) and dropout on projection.

class MaskEncoder(nn.Module):
    """
    512×512×1 binary mask → latent (B, latent_ch, 16, 16).
    Returns latent z AND skip features [f1, f2, f3, f4].

    Uses IDENTICAL DoubleConv+MaxPool blocks as UNet encoder.
    Channel progression: 64 → 128 → 256 → 512 → 512 (matches UNet exactly).

    Dropout2d before projection: forces structural representations,
    not memorisation of training masks (v3 fix).

    Skip channels: [64, 128, 256, 512]  (f1=256px, f2=128px, f3=64px, f4=32px)
    """
    SKIP_CHS = [64, 128, 256, 512]

    def __init__(self, cfg):
        super().__init__()
        self.stem  = DoubleConv(1, 64)          # 512×512,  64ch  (no pool)
        self.down1 = DownBlock(64,  64)          # 256×256,  64ch  → f1
        self.down2 = DownBlock(64,  128)         # 128×128, 128ch  → f2
        self.down3 = DownBlock(128, 256)         #  64×64,  256ch  → f3
        self.down4 = DownBlock(256, 512)         #  32×32,  512ch  → f4
        self.down5 = DownBlock(512, 512)         #  16×16,  512ch
        self.project = nn.Sequential(
            nn.Dropout2d(cfg.dropout),
            nn.Conv2d(512, cfg.latent_ch, 1, bias=False),
            nn.BatchNorm2d(cfg.latent_ch),
        )

    def forward(self, x):
        x  = self.stem(x)
        f1 = self.down1(x)
        f2 = self.down2(f1)
        f3 = self.down3(f2)
        f4 = self.down4(f3)
        x5 = self.down5(f4)
        z  = self.project(x5)
        return z, [f1, f2, f3, f4]


class ImageEncoder(nn.Module):
    """
    512×512×3 RGB image → latent (B, latent_ch, 16, 16).
    Mirrors MaskEncoder exactly — same DoubleConv, MaxPool, channel sizes.
    Critical: identical resolution pyramid so cross-modal skip injection works.

    Difference from MaskEncoder: input is 3ch (RGB).
    Skip channels: [64, 128, 256, 512]  — must equal MaskEncoder.SKIP_CHS.
    """
    SKIP_CHS = [64, 128, 256, 512]  # must equal MaskEncoder.SKIP_CHS

    def __init__(self, cfg):
        super().__init__()
        self.stem  = DoubleConv(3, 64)           # 512×512,  64ch
        self.down1 = DownBlock(64,  64)           # 256×256,  64ch  → f1
        self.down2 = DownBlock(64,  128)          # 128×128, 128ch  → f2
        self.down3 = DownBlock(128, 256)          #  64×64,  256ch  → f3
        self.down4 = DownBlock(256, 512)          #  32×32,  512ch  → f4
        self.down5 = DownBlock(512, 512)          #  16×16,  512ch
        self.project = nn.Sequential(
            nn.Dropout2d(cfg.dropout),
            nn.Conv2d(512, cfg.latent_ch, 1, bias=False),
            nn.BatchNorm2d(cfg.latent_ch),
        )

    def forward(self, x):
        x  = self.stem(x)
        f1 = self.down1(x)
        f2 = self.down2(f1)
        f3 = self.down3(f2)
        f4 = self.down4(f3)
        x5 = self.down5(f4)
        z  = self.project(x5)
        return z, [f1, f2, f3, f4]


print('MaskEncoder and ImageEncoder defined  (DoubleConv + MaxPool, ch: 64→128→256→512→512)')

In [ ]:
# Cell 4.2 — MaskDecoder with SkipFusionGates

class MaskDecoder(nn.Module):
    """
    Decodes latent (B, latent_ch, 16, 16) → mask logits (B, 1, 512, 512).

    SkipFusionGate injection at 4 resolutions (low→high):
      gate4 @ 32×32  : fuses f4/s4 (512ch)
      gate3 @ 64×64  : fuses f3/s3 (256ch)
      gate2 @ 128×128: fuses f2/s2 (128ch)
      gate1 @ 256×256: fuses f1/s1 (64ch)

    Stage 1: skips from MaskEncoder (same domain   → gates open wide)
    Stage 2: skips from ImageEncoder (cross-domain → gates selectively adapt)

    v2 CRITICAL FIX: gates are NOT frozen in Stage 2.
    Only upsampling weights (up1..up5, expand, out) stay frozen.
    """
    def __init__(self, cfg, skip_chs=None):
        super().__init__()
        if skip_chs is None:
            skip_chs = MaskEncoder.SKIP_CHS
        lch = cfg.latent_ch

        self.expand = nn.Sequential(nn.Conv2d(lch, 512, 1, bias=False), nn.BatchNorm2d(512))
        self.up1    = UpBlock(512, 512)                                # 16→32
        self.gate4  = SkipFusionGate(skip_chs[3], 512, 512)           # fuse f4
        self.up2    = UpBlock(512, 256)                                # 32→64
        self.gate3  = SkipFusionGate(skip_chs[2], 256, 256)           # fuse f3
        self.up3    = UpBlock(256, 128)                                # 64→128
        self.gate2  = SkipFusionGate(skip_chs[1], 128, 128)           # fuse f2
        self.up4    = UpBlock(128, 64)                                 # 128→256
        self.gate1  = SkipFusionGate(skip_chs[0], 64, 64)             # fuse f1
        self.up5    = UpBlock(64, 32)                                  # 256→512
        self.out    = nn.Conv2d(32, 1, 1)

    def forward(self, z, skips=None):
        x = self.expand(z)
        x = self.up1(x)
        if skips is not None: x = self.gate4(skips[3], x)
        x = self.up2(x)
        if skips is not None: x = self.gate3(skips[2], x)
        x = self.up3(x)
        if skips is not None: x = self.gate2(skips[1], x)
        x = self.up4(x)
        if skips is not None: x = self.gate1(skips[0], x)
        x = self.up5(x)
        return self.out(x)


print('MaskDecoder defined  (SkipFusionGate × 4, unfrozen in Stage 2)')

In [ ]:
# Cell 4.3 — LatentHead + MaskAutoencoder v3  (dual-path Stage 1)

class LatentHead(nn.Module):
    """
    v3 addition: direct latent supervision head used ONLY in Stage 1.

    Forces the MaskEncoder to produce a z that contains vessel shape
    information at the 16x16 latent resolution.

    Architecture:
      z (B, latent_ch, 16, 16)
        -> Upsample 2x (bilinear)          -> (B, latent_ch, 32, 32)
        -> Conv3x3 -> BN -> GELU           -> (B, 64, 32, 32)
        -> Conv3x3 -> BN -> ReLU           -> (B, 32, 32, 32)
        -> Conv1x1                         -> (B, 1,  32, 32)  logits

    Supervision: Lovász-Hinge at 32x32 (direct IoU optimisation on z).
    Gradient path: l_lat -> lat_pred -> z -> MaskEncoder (short, strong signal).
    Frozen in Stage 2 (freeze_for_stage2). Never called at inference.
    """
    def __init__(self, latent_ch):
        super().__init__()
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),  # 16->32
            nn.Conv2d(latent_ch, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),  # -> (B, 1, 32, 32)
        )

    def forward(self, z):
        return self.head(z)  # (B, 1, 32, 32) logits


class MaskAutoencoder(nn.Module):
    """
    v3 Stage 1 self-supervised module — three parallel training paths.

    All paths share a SINGLE encoder forward pass (z and skips computed once):

      Path 1 (main):    mask -> encoder -> (z, skips) -> decoder(z, skips) -> recon_full
                        Loss: L_pixel(recon_full, mask)         [lambda = 1.0]
                        Effect: decoder learns optimal z + skip fusion
                                gates learn from same-domain (mask->mask) skips

      Path 2 (z-only):  same z -> decoder(z, skips=None) -> recon_z_only
                        Loss: lambda_z * L_pixel(recon_z, mask) [lambda = 0.4]
                        Effect: decoder upsampling weights conditioned on z alone
                                encoder pressed to produce decodable z
                                NOTE: skips=None -> SkipFusionGates NOT called
                                      -> gate weights only receive grad from path 1

      Path 3 (latent):  same z -> LatentHead -> lat_pred (32x32)
                        Loss: lambda_lat * L_lovász(lat_pred, mask_32) [lambda = 0.3]
                        Effect: z directly supervised as a vessel map
                                short gradient path stays alive even at Dice=0.9999

    Critical property:
      Gate weights receive gradient ONLY from Path 1 (paths 2 and 3 bypass gates).
      Therefore gate learning is completely unaffected by v3 changes.
    """
    def __init__(self, cfg):
        super().__init__()
        self.encoder  = MaskEncoder(cfg)
        self.decoder  = MaskDecoder(cfg, skip_chs=MaskEncoder.SKIP_CHS)
        self.lat_head = LatentHead(cfg.latent_ch)   # v3 addition

    def forward(self, mask):
        # Single encoder forward — z and skips shared across all paths
        z, skips = self.encoder(mask)

        # Path 1: full reconstruction (z + skips) — existing behaviour
        recon_full   = self.decoder(z, skips)

        # Path 2: z-only reconstruction (skips=None, gates not called)
        recon_z_only = self.decoder(z, skips=None)

        # Path 3: direct latent prediction at 32x32
        lat_pred     = self.lat_head(z)

        return recon_full, recon_z_only, lat_pred, z, skips

    def freeze_for_stage2(self):
        """
        v3+ selective freezing:
          FROZEN:    MaskEncoder (all) + MaskDecoder upsampling + LatentHead
          TRAINABLE: SkipFusionGates 1-4 + expand conv  [v3+: expand unfrozen]

          Why unfreeze expand:
            expand is the first contact point between z_pred (image domain)
            and the frozen decoder (mask domain). Allowing it to adapt its
            512ch projection to image-latent statistics reduces domain shift
            for all subsequent frozen upsampling stages.
        """
        # Freeze entire MaskEncoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d): m.eval()

        # Freeze LatentHead (Stage 1 only, not needed in Stage 2)
        for p in self.lat_head.parameters():
            p.requires_grad = False

        # Freeze decoder upsampling, UNFREEZE gates + expand  [v3+]
        gate_params, expand_params, frozen_params = 0, 0, 0
        for name, p in self.decoder.named_parameters():
            if 'gate' in name or name.startswith('expand'):
                p.requires_grad = True
                if 'gate' in name: gate_params  += p.numel()
                else:              expand_params += p.numel()
            else:
                p.requires_grad = False
                frozen_params += p.numel()

        print(f'MaskEncoder + LatentHead frozen.')
        print(f'Decoder: {frozen_params:,} frozen | {gate_params:,} gate + {expand_params:,} expand trainable.')  # [v3+]


print('LatentHead + MaskAutoencoder v3 defined')
print('  Three paths: (z+skips)->recon | (z)->recon | (z)->32x32 latent head')


In [ ]:
# Cell 4.4 — SpatialMappingNet & AuxHead

class SpatialMappingNet(nn.Module):
    """
    Bridge: image latent → mask latent space.
    Both tensors: (B, latent_ch, 16, 16).

    Architecture:
      1×1 conv → channel expansion (per-position MLP)
      3×3 depthwise → spatial vessel continuity
      1×1 pointwise → mix channels
      project back + residual
    """
    def __init__(self, cfg):
        super().__init__()
        ch     = cfg.latent_ch      # 256
        hidden = ch * 2             # 512
        self.net = nn.Sequential(
            nn.Conv2d(ch, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden), nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),  # depthwise
            nn.Conv2d(hidden, hidden, 1, bias=False),                            # pointwise
            nn.BatchNorm2d(hidden), nn.GELU(),
            nn.Dropout2d(cfg.dropout),
            nn.Conv2d(hidden, ch, 1, bias=False),
            nn.BatchNorm2d(ch),
        )
        self.residual = nn.Conv2d(ch, ch, 1, bias=False)

    def forward(self, z):
        return self.net(z) + self.residual(z)




class GlobalLatentDNN(nn.Module):
    """
    Option 2: True Flattened Latent DNN (Global Multi-Layer Perceptron)
    Allows global spatial communication across all coordinates by flattening the 16×16 grid.
    """
    def __init__(self, cfg):
        super().__init__()
        self.latent_ch = cfg.latent_ch  # 256
        self.h_w = 16
        self.flat_dim = self.latent_ch * self.h_w * self.h_w  # 256 * 16 * 16 = 65,536

        # Deep fully connected neural network (DNN)
        self.dnn = nn.Sequential(
            nn.Linear(self.flat_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(cfg.dropout),

            nn.Linear(1024, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(cfg.dropout),

            nn.Linear(1024, self.flat_dim),
            nn.LayerNorm(self.flat_dim)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        # Flatten: (B, 256, 16, 16) -> (B, 65536)
        x_flat = x.view(B, -1)

        # Deep dense feed-forward
        out = self.dnn(x_flat)

        # Reshape back to spatial latent dimensions: (B, 256, 16, 16)
        return out.view(B, C, H, W) + x

class AuxHead(nn.Module):
    """
    Prediction AT the latent state (original design intent).
    z_pred (16×16×latent_ch) → upsample → conv head → logits (32×32).

    Uses Lovász-Hinge loss (same as main pixel loss — consistent supervision).
    Short gradient path: aux_loss → MappingNet → ImageEncoder.
    At inference: NOT called → zero overhead.
    """
    def __init__(self, cfg):
        super().__init__()
        ch = cfg.latent_ch
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(ch, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
        )  # 16→32

    def forward(self, z):
        return self.head(z)  # (B, 1, 32, 32)




In [ ]:
# Cell 4.5 — DualEncoderSegV2 (full model)

class DualEncoderSegV2(nn.Module):
    """
    DualEncoderSeg v2 — full segmentation model.

    Training forward (mask provided):
      image → ImageEncoder → (z_img, [f1..f4])
      z_img → MappingNet   → z_pred
      z_pred → AuxHead     → aux_logits (32×32)
      z_pred + [f1..f4] → MaskDecoder → pred_logits (512×512)
      mask  → MaskEncoder (frozen) → z_mask (for alignment loss)

    Inference forward (mask=None):
      image → ImageEncoder → z_img → MappingNet → z_pred
      z_pred + image_skips → MaskDecoder → logits
    """
    def __init__(self, cfg, pretrained_mae=None):
        super().__init__()
        self.cfg           = cfg
        self.image_encoder = ImageEncoder(cfg)
        if getattr(cfg, 'mapping_type', 'spatial') == 'global_dnn':
            self.mapping = GlobalLatentDNN(cfg)
        else:
            self.mapping = SpatialMappingNet(cfg)
        self.aux_head      = AuxHead(cfg)

        if pretrained_mae is not None:
            self.mask_decoder = pretrained_mae.decoder
            self.mask_encoder = pretrained_mae.encoder
            pretrained_mae.freeze_for_stage2()   # selective: encoder frozen, gates unfrozen
        else:
            self.mask_encoder = MaskEncoder(cfg)
            self.mask_decoder = MaskDecoder(cfg, skip_chs=ImageEncoder.SKIP_CHS)

    def forward(self, image, mask=None):
        z_img, img_skips = self.image_encoder(image)
        z_pred           = self.mapping(z_img)
        pred_logits      = self.mask_decoder(z_pred, img_skips)

        if mask is not None:
            aux_logits = self.aux_head(z_pred)
            with torch.no_grad():
                z_mask, _ = self.mask_encoder(mask)
            return pred_logits, aux_logits, z_pred, z_mask

        return pred_logits


def count_params(model, label=''):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{label:<28} total={total:>12,}  trainable={trainable:>12,}')
    return total, trainable




## §5 — UNet Baseline  *(identical conv blocks, same channel progression)*

In [ ]:
# Cell 5.1 — UNet Baseline (unchanged backbone — same DoubleConv+MaxPool as DualEncoder)

class UNetUp(nn.Module):
    """Bilinear 2× + concat skip + DoubleConv. Standard UNet decoder stage."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class UNetBaseline(nn.Module):
    """
    Standard UNet with 5 encoder stages.

    IDENTICAL to DualEncoder in:
      Conv block:    DoubleConv
      Downsampling:  MaxPool 2×2
      Channels:      64→128→256→512→512
      Image size:    512×512

    DIFFERENT from DualEncoder:
      Skip fusion:   plain concat + DoubleConv (no learned gate)
      Training:      1 stage, end-to-end
      Latent:        full 512ch bottleneck (not projected to 256ch)
    """
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.enc1       = DoubleConv(in_ch, 64)  # 512×512
        self.enc2       = DownBlock(64,  128)     # 256×256
        self.enc3       = DownBlock(128, 256)     # 128×128
        self.enc4       = DownBlock(256, 512)     #  64×64
        self.bottleneck = DownBlock(512, 512)     #  32×32
        self.dec4       = UNetUp(512, 512, 256)   #  64×64
        self.dec3       = UNetUp(256, 256, 128)   # 128×128
        self.dec2       = UNetUp(128, 128, 64)    # 256×256
        self.dec1       = UNetUp(64,   64, 32)    # 512×512
        self.out        = nn.Conv2d(32, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bottleneck(e4)
        d4 = self.dec4(b,  e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.out(d1)


# Quick existence check — full side-by-side breakdown is in the next cell
_d = DualEncoderSegV2(CFG); _u = UNetBaseline()
print(f'{"Model":<28} {"Total":>14}  {"Trainable":>14}')
print('-' * 60)
count_params(_d, 'DualEncoderSeg v3 (pre-freeze)')
count_params(_u, 'UNet Baseline')
del _d, _u
print('\nSee next cell for full per-module breakdown.')


In [ ]:
# Cell 5.2 — Side-by-side Parameter Count Comparison (v3)
# Shows: per-module breakdown, pre-freeze vs post-freeze Stage 2 trainable counts

import math

# ── Instantiate both models ──────────────────────────────────────────────────
_dual_full = DualEncoderSegV2(CFG)     # pre-freeze (all trainable)
_dual_mae  = MaskAutoencoder(CFG)      # for Stage 1 count
_unet      = UNetBaseline()

# ── Helper: count params in a named sub-module ───────────────────────────────
def module_params(model, attr_chain):
    """Navigate dotted attr path and count params. Returns (total, trainable)."""
    obj = model
    for attr in attr_chain.split('.'):
        obj = getattr(obj, attr)
    total     = sum(p.numel() for p in obj.parameters())
    trainable = sum(p.numel() for p in obj.parameters() if p.requires_grad)
    return total, trainable

# ── Compute post-freeze trainable count ─────────────────────────────────────
_dual_frozen = DualEncoderSegV2(CFG, pretrained_mae=MaskAutoencoder(CFG))
# (freeze_for_stage2 already called inside DualEncoderSegV2.__init__ via pretrained_mae)
s2_trainable = sum(p.numel() for p in _dual_frozen.parameters() if p.requires_grad)
s2_frozen    = sum(p.numel() for p in _dual_frozen.parameters() if not p.requires_grad)

# ── Print breakdown ──────────────────────────────────────────────────────────
W = 80
print('=' * W)
print(f'  PARAMETER COUNT — DualEncoderSeg v3 vs UNet Baseline')
print('=' * W)

# Header
print(f'  {"Component":<32} {"DualEnc v3":>14}  {"UNet":>14}  {"Notes"}')
print(f'  {"-"*32} {"-"*14}  {"-"*14}  {"-"*20}')

rows = [
    # (label, dual_attr, unet_attr_or_none, note)
    ('Image Encoder (stem+down1-5+proj)', 'image_encoder', None,
     'ImageEncoder (RGB)'),
    ('Mask Encoder (stem+down1-5+proj)',  'mask_encoder',  None,
     'FROZEN in Stage 2'),
    ('Mask Decoder (expand+up1-5+out)',   'mask_decoder',  None,
     'Upsampling FROZEN, gates trainable'),
    ('  └─ SkipFusionGates (×4)',         'mask_decoder',  None,
     'UNFROZEN in Stage 2'),
    ('Mapping Net (SpatialMap/GlobalDNN)','mapping',       None,
     'Trainable Stage 2'),
    ('AuxHead (Stage 2, 32×32)',          'aux_head',      None,
     'Trainable Stage 2'),
    ('LatentHead (Stage 1 only)',         None,            None,
     '[v3 NEW] FROZEN in Stage 2'),
    ('UNet Encoder (enc1-4+bottleneck)',  None,            None,
     'End-to-end'),
    ('UNet Decoder (dec1-4+out)',         None,            None,
     'End-to-end'),
]

# Individual counts
dual_img_enc   = sum(p.numel() for p in _dual_full.image_encoder.parameters())
dual_mask_enc  = sum(p.numel() for p in _dual_full.mask_encoder.parameters())
dual_dec_all   = sum(p.numel() for p in _dual_full.mask_decoder.parameters())
dual_gates     = sum(p.numel() for n,p in _dual_full.mask_decoder.named_parameters() if 'gate' in n)
dual_dec_up    = dual_dec_all - dual_gates
dual_mapping   = sum(p.numel() for p in _dual_full.mapping.parameters())
dual_aux       = sum(p.numel() for p in _dual_full.aux_head.parameters())
dual_lat_head  = sum(p.numel() for p in _dual_mae.lat_head.parameters())

unet_enc = sum(p.numel() for p in [*_unet.enc1.parameters(),
                                    *_unet.enc2.parameters(),
                                    *_unet.enc3.parameters(),
                                    *_unet.enc4.parameters(),
                                    *_unet.bottleneck.parameters()])
unet_dec = sum(p.numel() for p in [*_unet.dec4.parameters(),
                                    *_unet.dec3.parameters(),
                                    *_unet.dec2.parameters(),
                                    *_unet.dec1.parameters(),
                                    *_unet.out.parameters()])

breakdown = [
    ('Image Encoder',           f'{dual_img_enc:>14,}', f'{"—":>14}',      'Trainable Stage 2'),
    ('Mask Encoder',            f'{dual_mask_enc:>14,}', f'{"—":>14}',     'FROZEN in Stage 2'),
    ('Mask Decoder (all)',       f'{dual_dec_all:>14,}', f'{"—":>14}',     ''),
    ('  ├─ Upsampling (frozen)', f'{dual_dec_up:>14,}',  f'{"—":>14}',    'FROZEN in Stage 2'),
    ('  └─ SkipFusionGates ×4', f'{dual_gates:>14,}',   f'{"—":>14}',    'UNFROZEN in Stage 2'),
    ('MappingNet',              f'{dual_mapping:>14,}',  f'{"—":>14}',     'Trainable Stage 2'),
    ('AuxHead (Stage 2)',        f'{dual_aux:>14,}',     f'{"—":>14}',     'Trainable Stage 2'),
    ('LatentHead (Stage 1 only)',f'{dual_lat_head:>14,}',f'{"—":>14}',    '[v3] FROZEN in Stage 2'),
    ('─'*32,                    '─'*14,                  '─'*14,          ''),
    ('UNet Encoder',            f'{"—":>14}',            f'{unet_enc:>14,}', 'Trainable'),
    ('UNet Decoder',            f'{"—":>14}',            f'{unet_dec:>14,}', 'Trainable'),
]

for label, dual_val, unet_val, note in breakdown:
    print(f'  {label:<32} {dual_val}  {unet_val}  {note}')

# ── Totals ───────────────────────────────────────────────────────────────────
dual_total      = sum(p.numel() for p in _dual_full.parameters())
unet_total      = sum(p.numel() for p in _unet.parameters())
mae_total       = sum(p.numel() for p in _dual_mae.parameters())
ratio           = dual_total / unet_total

print()
print('=' * W)
print(f'  {"TOTALS":<32} {"DualEnc v3":>14}  {"UNet":>14}  {"Notes"}')
print(f'  {"-"*32} {"-"*14}  {"-"*14}  {"-"*20}')
print(f'  {"Stage 1 MAE (pre-freeze)":<32} {mae_total:>14,}  {"—":>14}  MaskEncoder+Decoder+LatentHead')
print(f'  {"Stage 2 full model":<32} {dual_total:>14,}  {unet_total:>14,}  Both trainable (pre-freeze)')
print(f'  {"Stage 2 trainable (post-freeze)":<32} {s2_trainable:>14,}  {unet_total:>14,}  DualEnc after freeze_for_stage2()')
print(f'  {"Stage 2 frozen":<32} {s2_frozen:>14,}  {"0":>14}  DualEnc frozen params')
print(f'  {"Model size ratio (pre-freeze)":<32} {ratio:>14.2f}×  {"1.00×":>14}  DualEnc/UNet')
print()
print(f'  NOTE: with mapping_type="spatial" (SpatialMappingNet), DualEnc v3 is')
print(f'  comparable to UNet in total params. Use mapping_type="global_dnn" only')
print(f'  if you have a large dataset (overfits on small datasets).')
print('=' * W)

del _dual_full, _dual_mae, _dual_frozen, _unet


## §6 — Complex Loss Suite  *(shared + DualEnc-specific)*

In [ ]:
# Cell 6.1 — Shared pixel losses (applied identically to both models)

def tversky_loss(logits, target, alpha=0.3, beta=0.7, smooth=1e-5):
    """
    Asymmetric Dice (Tversky). β > α → missing vessels penalised more than FP.
    Identical weights used for both DualEncoder and UNet.
    """
    pred = torch.sigmoid(logits)
    tp   = (pred * target).sum(dim=(2, 3))
    fp   = (pred * (1 - target)).sum(dim=(2, 3))
    fn   = ((1 - pred) * target).sum(dim=(2, 3))
    tv   = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return (1 - tv).mean()


def boundary_loss(logits, target):
    """
    Sobel-weighted BCE. Applies 5× weight at vessel boundary pixels.
    Identical for both models.
    """
    sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]],
                            device=target.device).view(1, 1, 3, 3)
    sobel_y = sobel_x.transpose(2, 3)
    edge    = (F.conv2d(target, sobel_x, padding=1).abs() +
               F.conv2d(target, sobel_y, padding=1).abs()).clamp(0, 1)
    bce     = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    return (bce * (1 + 4 * edge)).mean()


def focal_loss(logits, target, gamma=2.0, alpha=0.25):
    """
    Binary Focal Loss — hard example mining.
    Down-weights easy background, forces focus on ambiguous vessel pixels.
    Applied identically to both models.

    α=0.25: vessel class (positive) given lower weight than background
            because vessel pixels are rare but focal already up-weights hard ones.
    γ=2.0:  standard modulating exponent.
    """
    bce     = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    p_t     = torch.exp(-bce)                                          # p(correct class)
    alpha_t = alpha * target + (1 - alpha) * (1 - target)             # per-pixel alpha
    return (alpha_t * (1 - p_t) ** gamma * bce).mean()


print('Shared losses defined: tversky_loss, boundary_loss, focal_loss')

In [ ]:
# Cell 6.2 — Lovász-Hinge Loss (direct IoU optimisation)

def _lovasz_grad(gt_sorted):
    """
    Gradient of the Lovász extension w.r.t sorted errors.
    This is the piece-wise linear approximation of the Jaccard (IoU) loss.
    """
    p             = len(gt_sorted)
    gts           = gt_sorted.sum()
    intersection  = gts - gt_sorted.float().cumsum(0)
    union         = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard       = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def _lovasz_hinge_flat(logits, labels):
    """Binary Lovász hinge on flat (1-D) logits and integer labels."""
    if len(labels) == 0:
        return logits.sum() * 0.
    signs         = 2. * labels.float() - 1.          # +1 for vessel, -1 for background
    errors        = 1. - logits * signs                # hinge errors
    errors_sorted, perm = torch.sort(errors, descending=True)
    gt_sorted     = labels[perm.data]
    grad          = _lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


def lovasz_hinge(logits, labels, per_image=True):
    """
    Binary Lovász-Hinge Loss — directly optimises IoU.

    Why Lovász is ideal for vessel segmentation:
      - Dice can plateau on thin/disconnected vessels (it's a ratio, not a ranking)
      - Lovász directly minimises the Jaccard error (1 - IoU) which is the
        actual evaluation metric
      - For DualEncoder: AuxHead uses Lovász at 32×32 resolution, providing
        a consistent gradient signal at the latent prediction level

    logits: (B, 1, H, W) — raw scores (not sigmoid)
    labels: (B, 1, H, W) — binary {0, 1}
    """
    if logits.dim() == 4: logits = logits.squeeze(1)     # (B, H, W)
    if labels.dim() == 4: labels = labels.squeeze(1)     # (B, H, W)

    B = logits.shape[0]
    if per_image:
        losses = [_lovasz_hinge_flat(logits[b].view(-1),
                                     labels[b].view(-1).long()) for b in range(B)]
        return torch.stack(losses).mean()
    else:
        return _lovasz_hinge_flat(logits.view(-1), labels.view(-1).long())


print('Lovász-Hinge loss defined')

In [ ]:
# Cell 6.3 — DualEncoder-specific alignment loss (replaces VICReg)

def latent_alignment_loss(z_pred, z_mask, lambda_mse=1.0, lambda_cos=1.0):
    """
    Replaces VICReg from v3.

    Why not VICReg:
      VICReg covariance term needs B >> C degrees of freedom.
      At B=4 with C=256 channels, the estimate has 3 d.f. — pure noise.

    This loss:
      MSE:    pulls z_pred toward z_mask in Euclidean space
      Cosine: aligns z_pred direction with z_mask direction
      Both are statistically valid at any batch size including B=1.

    z_pred, z_mask: (B, latent_ch, H, W) spatial latent tensors
    """
    # MSE alignment (magnitude + direction)
    l_mse = F.mse_loss(z_pred, z_mask.detach())

    # Cosine alignment (direction only, per spatial position)
    cos_sim = F.cosine_similarity(z_pred, z_mask.detach(), dim=1)  # (B, H, W)
    l_cos   = (1. - cos_sim).mean()

    return lambda_mse * l_mse + lambda_cos * l_cos


def get_alignment_weight(epoch, stage2_epochs):
    """
    Curriculum schedule for alignment loss weight.  [v3+: slower decay]
    Gradually yields control to pixel losses as training progresses.

    Epoch  1–20:  1.0  (establish rough latent alignment)
    Epoch 21–60:  0.5  (pixel loss starts taking over)
    Epoch 61–90:  0.3  (slower decay — prevents mapping net drift)  [v3+ NEW]
    Epoch 91+:    0.2  (pixel loss dominates, latent stays anchored)

    Rationale: pixel loss total ≈ 4.0× (Tversky+Boundary+Focal+Lovász).
    Old schedule dropped to 0.2 at epoch 61, making alignment only 5%
    of the total signal — allowing z_pred to drift from z_mask.
    Adding an intermediate 0.3 step keeps alignment ≈ 7% for 30 more
    epochs before the final decay, without sacrificing pixel loss dominance.
    """
    if epoch <= 20:
        return 1.0
    elif epoch <= 60:
        return 0.5
    elif epoch <= 90:   # [v3+] intermediate step — slower decay
        return 0.3
    else:
        return 0.2


print('Latent alignment loss defined  (MSE + Cosine, curriculum: 1.0→0.5→0.3→0.2)  [v3+]')

In [ ]:
# Cell 6.4 — Combined loss modules

class PixelLoss(nn.Module):
    """
    Shared complex pixel loss — applied IDENTICALLY to both models.
    4 terms: Tversky + Boundary + Focal + Lovász

    Each term targets a different failure mode of thin-vessel segmentation:
      Tversky:  recall bias (β > α, punishes missing vessels)
      Boundary: edge precision (5× weight at Sobel-detected vessel edges)
      Focal:    hard example mining (γ=2, focuses on ambiguous pixels)
      Lovász:   direct IoU optimisation (smooth surrogate of Jaccard)
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def forward(self, logits, target):
        l_tvk = tversky_loss(logits, target, self.cfg.tversky_alpha, self.cfg.tversky_beta)
        l_bnd = boundary_loss(logits, target)
        l_fcl = focal_loss(logits, target, self.cfg.focal_gamma, self.cfg.focal_alpha)
        l_lov = lovasz_hinge(logits, target)
        total = (self.cfg.lambda_tversky  * l_tvk
               + self.cfg.lambda_boundary * l_bnd
               + self.cfg.lambda_focal    * l_fcl
               + self.cfg.lambda_lovasz   * l_lov)
        breakdown = {'tversky': l_tvk.item(), 'boundary': l_bnd.item(),
                     'focal': l_fcl.item(), 'lovasz': l_lov.item()}
        return total, breakdown


class DualEncoderLoss(nn.Module):
    """
    Full Stage 2 loss for DualEncoderSeg v2.
    = PixelLoss (shared 4-term) + latent alignment + aux head Lovász.
    """
    def __init__(self, cfg, align_weight=1.0):
        super().__init__()
        self.pixel  = PixelLoss(cfg)
        self.cfg    = cfg
        self.lw     = align_weight

    def forward(self, pred_logits, aux_logits, z_pred, z_mask, target):
        l_pixel, bd = self.pixel(pred_logits, target)

        # Alignment: z_pred ↔ z_mask (MSE + Cosine, curriculum weighted)
        l_align = latent_alignment_loss(z_pred, z_mask,
                                        self.cfg.lambda_align_mse,
                                        self.cfg.lambda_align_cos)

        # Aux head: Lovász at 32×32 (bilinear downsample GT — not nearest-neighbor)
        aux_target = F.interpolate(target, size=(32, 32), mode='bilinear', align_corners=False)
        l_aux      = lovasz_hinge(aux_logits, (aux_target > 0.5).float())

        total = l_pixel + self.lw * l_align + self.cfg.lambda_aux * l_aux

        bd['align'] = l_align.item()
        bd['aux']   = l_aux.item()
        bd['align_w'] = self.lw
        return total, bd


class Stage1Loss(nn.Module):
    """
    v3 Stage 1: three-term loss for dual-path MaskAutoencoder.

    L_total = L_pixel(recon_full, mask)              path 1: z + skips (main)
            + lambda_z   * L_pixel(recon_z, mask)    path 2: z alone (z-only)
            + lambda_lat * L_lovász(lat_pred, mask32) path 3: direct z supervision

    Path 1 trains the full skip-injection pipeline (gates unaffected by paths 2/3).
    Path 2 forces the decoder upsampling to be meaningfully conditioned on z.
    Path 3 directly supervises z to encode vessel shape (short, strong gradient).
    Paths 2+3 keep gradients alive when path 1 saturates at Dice=0.9999.
    """
    def __init__(self, cfg):
        super().__init__()
        self.pixel      = PixelLoss(cfg)
        self.lambda_z   = cfg.lambda_z
        self.lambda_lat = cfg.lambda_lat

    def forward(self, recon_full, recon_z_only, lat_pred, target):
        # Path 1: full reconstruction (z + skips)
        l_main,   _ = self.pixel(recon_full,   target)

        # Path 2: z-only reconstruction (decoder without skips)
        l_z_only, _ = self.pixel(recon_z_only, target)

        # Path 3: direct latent supervision at 32x32
        target_32 = F.interpolate(target, size=(32, 32),
                                  mode='bilinear', align_corners=False)
        target_32 = (target_32 > 0.5).float()
        l_lat = lovasz_hinge(lat_pred, target_32)

        total = l_main + self.lambda_z * l_z_only + self.lambda_lat * l_lat
        return total, {
            'main':   l_main.item(),
            'z_only': l_z_only.item(),
            'lat':    l_lat.item(),
        }


print('Loss modules: PixelLoss (shared 4-term), DualEncoderLoss, Stage1Loss v3 (3-term)')

## §7 — Metrics

In [ ]:
# Cell 7.1 — compute_metrics (identical for both models)

@torch.no_grad()
def compute_metrics(logits, target, threshold=0.5):
    """IoU, Dice, Accuracy, Sensitivity, Specificity. Same function for both models."""
    pred = (torch.sigmoid(logits) > threshold).float()
    tp = (pred * target).sum(dim=(1, 2, 3))
    fp = (pred * (1 - target)).sum(dim=(1, 2, 3))
    fn = ((1 - pred) * target).sum(dim=(1, 2, 3))
    tn = ((1 - pred) * (1 - target)).sum(dim=(1, 2, 3))
    s  = 1e-5
    return {
        'iou':         ((tp + s) / (tp + fp + fn + s)).mean().item(),
        'dice':        ((2 * tp + s) / (2 * tp + fp + fn + s)).mean().item(),
        'acc':         (pred == target).float().mean().item(),
        'sensitivity': ((tp + s) / (tp + fn + s)).mean().item(),
        'specificity': ((tn + s) / (tn + fp + s)).mean().item(),
    }


print('compute_metrics defined  (IoU, Dice, Accuracy, Sensitivity, Specificity)')

## §8 — Dataset & DataLoaders

In [ ]:
# Cell 8.1 — RetinaDataset (identical augmentation for both models)

class RetinaDataset(Dataset):
    """Same augmentation pipeline applied to both training runs."""
    def __init__(self, image_paths, mask_paths, augment=False, image_size=512):
        assert len(image_paths) == len(mask_paths)
        self.image_paths = list(image_paths)
        self.mask_paths  = list(mask_paths)
        self.augment     = augment
        self.size        = image_size

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        from PIL import Image
        import torchvision.transforms.functional as TF
        img  = Image.open(self.image_paths[idx]).convert('RGB').resize((self.size, self.size))
        mask = Image.open(self.mask_paths[idx]).convert('L').resize((self.size, self.size))
        if self.augment:
            if random.random() > 0.5: img, mask = TF.hflip(img), TF.hflip(mask)
            if random.random() > 0.5: img, mask = TF.vflip(img), TF.vflip(mask)
            angle = random.uniform(-30, 30)
            img   = TF.rotate(img,  angle, fill=0)
            mask  = TF.rotate(mask, angle, fill=0)
            img   = TF.adjust_brightness(img, random.uniform(0.7, 1.3))
            img   = TF.adjust_contrast(img,   random.uniform(0.7, 1.3))
            img   = TF.adjust_saturation(img, random.uniform(0.7, 1.3))
            if random.random() > 0.7: img = TF.gaussian_blur(img, kernel_size=3)
        img  = TF.to_tensor(img)
        mask = TF.to_tensor(mask)
        if mask.max() > 0:
            mask = mask / mask.max()
        mask = (mask > 0.5).float()
        return img, mask


print('RetinaDataset defined')

In [ ]:
_pin = torch.cuda.is_available()  # only useful with a GPU
# Cell 8.2 — Build DataLoaders (set DATA_ROOT in §1)

data_root  = Path(CFG.DATA_ROOT)
img_paths  = sorted((data_root / 'image').glob('*.png'))
mask_paths = sorted((data_root / 'mask').glob('*.png'))
assert len(img_paths) > 0, f'No images found at {data_root / "image"}'
assert len(img_paths) == len(mask_paths), 'Image/mask count mismatch'

split        = int(CFG.train_split * len(img_paths))
train_imgs   = img_paths[:split];   val_imgs   = img_paths[split:]
train_masks  = mask_paths[:split];  val_masks  = mask_paths[split:]

train_dl = DataLoader(RetinaDataset(train_imgs, train_masks, augment=True,  image_size=CFG.image_size),
                      batch_size=CFG.batch_size, shuffle=True,  num_workers=CFG.num_workers, pin_memory=_pin)
val_dl   = DataLoader(RetinaDataset(val_imgs,   val_masks,   augment=False, image_size=CFG.image_size),
                      batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=_pin)

print(f'Train: {len(train_imgs)} samples ({len(train_dl)} batches)  |  Val: {len(val_imgs)} samples ({len(val_dl)} batches)')

## §9 — Training Functions

In [ ]:
# Cell 9.0 — Shared LR scheduler

def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    """Linear warmup → cosine decay. Identical for both models."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


print('Scheduler defined')

In [ ]:
# Cell 9.1 — train_stage1 v3: dual-path MaskAutoencoder pretraining

def train_stage1(mae, train_loader, val_loader=None, cfg=None, device='cuda'):
    """
    v3 Stage 1 — dual-path pretraining.

    Loss: L_main + lambda_z*L_z_only + lambda_lat*L_lat

      L_main   : full path (z + skips -> recon) — teaches skip + z joint decoding
      L_z_only : z-only path (z -> recon, no skips) — forces decoder z-conditioning
      L_lat    : latent head (z -> 32x32) — directly supervises z representation

    Key properties:
      - Single encoder forward pass per batch (z, skips computed once)
      - Two decoder forward passes (with skips, without skips)
      - Gates receive gradient ONLY from L_main (L_z_only uses skips=None)
      - Gradient stays non-zero throughout all 60 epochs (L_z_only + L_lat never saturate)
      - No architecture changes to MaskEncoder / MaskDecoder / SkipFusionGate
    """
    if cfg is None: cfg = CFG
    mae       = mae.to(device)
    criterion = Stage1Loss(cfg)
    optimizer = AdamW(mae.parameters(), lr=cfg.stage1_lr, weight_decay=1e-4)
    scheduler = get_warmup_cosine_scheduler(optimizer, cfg.warmup_epochs, cfg.stage1_epochs)
    history   = {
        'train_loss': [], 'val_dice': [], 'val_iou': [], 'val_sens': [],
        'main': [], 'z_only': [], 'lat': [],   # per-term tracking (v3)
    }

    n_params = sum(p.numel() for p in mae.parameters())
    print('=' * 65)
    print('STAGE 1 v3 -- MaskAutoencoder dual-path  (full run, no early stop)')
    print(f'  Params: {n_params:,}')
    print(f'  Loss: L_main + {cfg.lambda_z}*L_z_only + {cfg.lambda_lat}*L_lat')
    print(f'  Paths: (z+skips->recon) + (z->recon) + (z->32x32 latent head)')
    print('=' * 65)

    pbar = tqdm(range(1, cfg.stage1_epochs + 1), desc='Stage 1 v3')
    for epoch in pbar:
        mae.train()
        total_loss = 0.0
        bd_acc     = {'main': 0., 'z_only': 0., 'lat': 0.}

        for batch in train_loader:
            masks = (batch[1] if isinstance(batch, (list, tuple)) else batch).float().to(device)
            optimizer.zero_grad()

            # Dual-path forward (single encoder, two decoder calls, one latent head call)
            recon_full, recon_z_only, lat_pred, z, skips = mae(masks)

            # Three-term loss
            loss, bd = criterion(recon_full, recon_z_only, lat_pred, masks)
            loss.backward()
            nn.utils.clip_grad_norm_(mae.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            for k in bd_acc: bd_acc[k] += bd[k]

        scheduler.step()
        n   = len(train_loader)
        avg = total_loss / n
        history['train_loss'].append(avg)
        for k in bd_acc: history[k].append(bd_acc[k] / n)

        if val_loader is not None:
            mae.eval()
            dl, il, sl = [], [], []
            with torch.no_grad():
                for batch in val_loader:
                    masks_v = (batch[1] if isinstance(batch, (list, tuple)) else batch).float().to(device)
                    # Validate on full path (z + skips) — primary reconstruction quality
                    recon_v, _, _, _, _ = mae(masks_v)
                    m = compute_metrics(recon_v, masks_v)
                    dl.append(m['dice']); il.append(m['iou']); sl.append(m['sensitivity'])
            vd, vi, vs = np.mean(dl), np.mean(il), np.mean(sl)
            history['val_dice'].append(vd)
            history['val_iou'].append(vi)
            history['val_sens'].append(vs)
            pbar.set_postfix({
                'loss':   f'{avg:.4f}',
                'Dice':   f'{vd:.4f}',
                'z_only': f'{bd_acc["z_only"]/n:.4f}',
                'lat':    f'{bd_acc["lat"]/n:.4f}',
            })

    last_dice = history['val_dice'][-1] if history['val_dice'] else float('nan')
    print(f'Stage 1 v3 complete. Final full-path Val Dice: {last_dice:.4f}')
    print(f'  (z-only + lat losses prevent gradient death -- expect Dice < 0.9999)')
    print()
    return history


print('train_stage1 v3 defined')


In [ ]:
# Cell 9.2 — train_stage2: DualEncoderSegV2 full training

def train_stage2(model, train_loader, val_loader=None, cfg=None, device='cuda'):
    """
    Stage 2: DualEncoderSegV2.
    MaskEncoder frozen. Decoder upsampling frozen. SkipFusionGates UNFROZEN.
    Alignment weight follows curriculum schedule.
    NO early stopping.
    """
    if cfg is None: cfg = CFG
    model     = model.to(device)
    # [v3+] Separate LR groups: gates get 2× LR for faster cross-domain adaptation
    gate_params  = [p for n, p in model.named_parameters()
                    if p.requires_grad and 'gate' in n]
    other_params = [p for n, p in model.named_parameters()
                    if p.requires_grad and 'gate' not in n]
    trainable = gate_params + other_params
    optimizer = AdamW(
        [{'params': gate_params,  'lr': cfg.stage2_lr * 2.0},   # gates: 2× LR  [v3+]
         {'params': other_params, 'lr': cfg.stage2_lr}],
        weight_decay=1e-4,
    )
    scheduler = get_warmup_cosine_scheduler(optimizer, cfg.warmup_epochs, cfg.stage2_epochs)
    history   = {'train_loss': [], 'val_dice': [], 'val_iou': [], 'val_sens': [],
                 'tversky': [], 'boundary': [], 'focal': [], 'lovasz': [],
                 'align': [], 'aux': [], 'align_w': []}

    trainable_n = sum(p.numel() for p in trainable)
    frozen_n    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print('=' * 65)
    print('STAGE 2 — DualEncoderSeg v3  (gates unfrozen, full run)')
    print(f'  Trainable: {trainable_n:,}  |  Frozen: {frozen_n:,}')
    print(f'  Loss: Tversky + Boundary + Focal + Lovász + Align + Aux')
    print('=' * 65)

    pbar = tqdm(range(1, cfg.stage2_epochs + 1), desc='Stage 2')
    for epoch in pbar:
        lw        = get_alignment_weight(epoch, cfg.stage2_epochs)
        criterion = DualEncoderLoss(cfg, align_weight=lw)

        model.train()
        # Keep frozen parts fixed
        model.mask_encoder.eval()
        for m in model.mask_encoder.modules():
            if isinstance(m, nn.BatchNorm2d): m.eval()
        # Decoder upsampling BN also stays in eval (was frozen)
        for name, m in model.mask_decoder.named_modules():
            if isinstance(m, nn.BatchNorm2d) and 'gate' not in name:
                m.eval()

        total_loss = 0.0
        bd_acc = {'tversky': 0., 'boundary': 0., 'focal': 0., 'lovasz': 0., 'align': 0., 'aux': 0.}

        for images, masks in train_loader:
            images = images.float().to(device); masks = masks.float().to(device)
            optimizer.zero_grad()
            pred_logits, aux_logits, z_pred, z_mask = model(images, masks)
            loss, bd = criterion(pred_logits, aux_logits, z_pred, z_mask, masks)
            loss.backward()
            nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            total_loss += loss.item()
            for k in bd_acc:
                if k in bd: bd_acc[k] += bd[k]

        scheduler.step()
        n     = len(train_loader)
        avg   = total_loss / n
        history['train_loss'].append(avg)
        history['align_w'].append(lw)
        for k in bd_acc: history[k].append(bd_acc[k] / n)

        if val_loader is not None:
            model.eval()
            dl, il, sl = [], [], []
            with torch.no_grad():
                for images, masks in val_loader:
                    images = images.float().to(device); masks = masks.float().to(device)
                    pred = model(images)
                    m = compute_metrics(pred, masks)
                    dl.append(m['dice']); il.append(m['iou']); sl.append(m['sensitivity'])
            vd, vi, vs = np.mean(dl), np.mean(il), np.mean(sl)
            history['val_dice'].append(vd); history['val_iou'].append(vi); history['val_sens'].append(vs)
            pbar.set_postfix({'loss': f'{avg:.4f}', 'Dice': f'{vd:.4f}', 'lw': f'{lw:.1f}'})

    print('Stage 2 complete.\n')
    return history


print('train_stage2 defined  [v3+: gate params use 2× LR, slower alignment curriculum]')

In [ ]:
# Cell 9.3 — train_unet: UNet Baseline single-stage training

def train_unet(model, train_loader, val_loader=None, cfg=None, device='cuda'):
    """
    UNet Baseline end-to-end training.
    Uses same 4-term PixelLoss as DualEncoder Stage 2 (controlled comparison).
    NO early stopping.
    """
    if cfg is None: cfg = CFG
    model     = model.to(device)
    criterion = PixelLoss(cfg)
    optimizer = AdamW(model.parameters(), lr=cfg.unet_lr, weight_decay=1e-4)
    scheduler = get_warmup_cosine_scheduler(optimizer, cfg.warmup_epochs, cfg.unet_epochs)
    history   = {'train_loss': [], 'val_dice': [], 'val_iou': [], 'val_sens': [],
                 'tversky': [], 'boundary': [], 'focal': [], 'lovasz': []}

    total_p = sum(p.numel() for p in model.parameters())
    print('=' * 65)
    print('UNet Baseline  (full run, no early stop)')
    print(f'  Total params: {total_p:,}')
    print(f'  Loss: Tversky + Boundary + Focal + Lovász  (same as DualEnc)')
    print('=' * 65)

    pbar = tqdm(range(1, cfg.unet_epochs + 1), desc='UNet')
    for epoch in pbar:
        model.train()
        total_loss = 0.0
        bd_acc = {'tversky': 0., 'boundary': 0., 'focal': 0., 'lovasz': 0.}
        for images, masks in train_loader:
            images = images.float().to(device); masks = masks.float().to(device)
            optimizer.zero_grad()
            pred = model(images)
            loss, bd = criterion(pred, masks)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            for k in bd_acc: bd_acc[k] += bd[k]

        scheduler.step()
        n   = len(train_loader)
        avg = total_loss / n
        history['train_loss'].append(avg)
        for k in bd_acc: history[k].append(bd_acc[k] / n)

        if val_loader is not None:
            model.eval()
            dl, il, sl = [], [], []
            with torch.no_grad():
                for images, masks in val_loader:
                    images = images.float().to(device); masks = masks.float().to(device)
                    pred = model(images)
                    m = compute_metrics(pred, masks)
                    dl.append(m['dice']); il.append(m['iou']); sl.append(m['sensitivity'])
            vd, vi, vs = np.mean(dl), np.mean(il), np.mean(sl)
            history['val_dice'].append(vd); history['val_iou'].append(vi); history['val_sens'].append(vs)
            pbar.set_postfix({'loss': f'{avg:.4f}', 'Dice': f'{vd:.4f}', 'IoU': f'{vi:.4f}'})

    print('UNet training complete.\n')
    return history


print('train_unet defined')

## §10 — Run Training

In [ ]:
# Cell 10.1 — Train Stage 1: MaskAutoencoder v3 (dual-path)
mae   = MaskAutoencoder(CFG)
hist1 = train_stage1(mae, train_dl, val_dl, cfg=CFG, device=CFG.device)


In [ ]:
# Cell 10.2 — Train Stage 2: DualEncoderSegV2 (with v3-pretrained MAE)
dual_model = DualEncoderSegV2(CFG, pretrained_mae=mae)
print('\nParam counts after freeze_for_stage2():')
count_params(dual_model, 'DualEncoderSeg v3 Stage2')
hist2 = train_stage2(dual_model, train_dl, val_dl, cfg=CFG, device=CFG.device)


In [ ]:
# Cell 10.3 — Train UNet Baseline
unet_model = UNetBaseline()
hist_unet  = train_unet(unet_model, train_dl, val_dl, cfg=CFG, device=CFG.device)

In [ ]:
# Cell 10.4 — Save checkpoints
torch.save({'model_state': dual_model.state_dict(), 'cfg': asdict(CFG),
            'hist1': hist1, 'hist2': hist2},
           f'{CFG.save_dir}/dual_encoder_v3.pt')
torch.save({'model_state': unet_model.state_dict(), 'cfg': asdict(CFG),
            'hist_unet': hist_unet},
           f'{CFG.save_dir}/unet_baseline_v2.pt')
print(f'Saved → {CFG.save_dir}/dual_encoder_v3.pt')
print(f'Saved → {CFG.save_dir}/unet_baseline_v2.pt')

## §11 — Results & Comparison Plots

In [ ]:
# Cell 11.0 — Plot helpers
DUAL_COLOR = '#2196F3'; UNET_COLOR = '#FF5722'; S1_COLOR = '#4CAF50'

def smooth(vals, w=5):
    if len(vals) < w: return vals
    kernel = np.ones(w) / w
    return np.convolve(np.pad(vals, (w//2, w//2), 'edge'), kernel, 'valid')[:len(vals)]

print('Plot helpers defined  |  DualEnc=Blue  UNet=Orange')

In [ ]:
# Cell 11.1 — Full validation metric evaluation
# v3 fix: accumulate all logits and masks, compute metrics once on full set.
# v2 was averaging per-batch (micro-batch bias with batch_size=4).

def evaluate_model(model, val_loader, device, name):
    """
    Correct evaluation: collect all predictions across the full validation set,
    then call compute_metrics once. This removes the micro-batch averaging
    bias present in v2 (where batch_size=4 micro-metrics were mean-averaged).
    """
    model.eval()
    all_logits, all_masks = [], []
    with torch.no_grad():
        for images, masks in val_loader:
            logits = model(images.float().to(device))
            all_logits.append(logits.cpu())
            all_masks.append(masks.cpu())
    all_logits = torch.cat(all_logits)   # (N, 1, H, W)
    all_masks  = torch.cat(all_masks)    # (N, 1, H, W)
    means = compute_metrics(all_logits, all_masks)
    print(f'\n{name} -- Final Validation  (n={all_logits.shape[0]} images)')
    print('-' * 44)
    for k, v in means.items(): print(f'  {k:<16}: {v:.4f}')
    return means


dual_final = evaluate_model(dual_model, val_dl, CFG.device, 'DualEncoderSeg v3')
unet_final = evaluate_model(unet_model, val_dl, CFG.device, 'UNet Baseline')

print('\n=== HEAD-TO-HEAD SUMMARY ===')
print(f'  {"Metric":<18} {"DualEnc v3":>12}  {"UNet":>10}  {"Delta":>10}')
print('  ' + '-' * 52)
for k in dual_final:
    d = dual_final[k] - unet_final[k]
    sign = '+' if d >= 0 else ''
    print(f'  {k:<18} {dual_final[k]:>12.4f}  {unet_final[k]:>10.4f}  {sign}{d:>9.4f}')


In [ ]:
# Cell 11.2 — Training loss curves (v3: Stage 1 shows 3-term breakdown)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Training Loss Curves -- DualEncoderSeg v3', fontsize=14, fontweight='bold')

# Stage 1 — three-term breakdown
ax = axes[0]
ax.set_title('Stage 1 -- MaskAutoencoder v3 (3-term loss)', fontsize=11)
if hist1.get('main'):
    ax.plot(smooth(hist1['main']),        color='#4CAF50', lw=2,         label='L_main (z+skips)')
    ax.plot(smooth(hist1['z_only']),      color='#FF9800', lw=2,         label='L_z_only (z alone)')
    ax.plot(smooth(hist1['lat']),         color='#9C27B0', lw=2,         label='L_lat (32x32 latent)')
    ax.plot(smooth(hist1['train_loss']),  color='#333',   lw=1.5, ls='--', label='Total')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Stage 2
ax = axes[1]
ax.set_title('Stage 2 -- DualEncoderSeg v3', fontsize=11)
if hist2.get('train_loss'):
    ax.plot(smooth(hist2['train_loss']), color=DUAL_COLOR, lw=2, label='DualEnc v3 Train')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# UNet baseline
ax = axes[2]
ax.set_title('UNet Baseline', fontsize=11)
if hist_unet.get('train_loss'):
    ax.plot(smooth(hist_unet['train_loss']), color=UNET_COLOR, lw=2, label='UNet Train')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v3_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 11.3 — Val Dice & IoU over epochs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Validation Metrics over Epochs -- DualEncoderSeg v3', fontsize=14, fontweight='bold')

for ax, metric, title in zip(axes, ['val_dice', 'val_iou'], ['Val Dice', 'Val IoU']):
    if hist2.get(metric):    ax.plot(hist2[metric],    color=DUAL_COLOR, lw=2,       label='DualEncoderSeg v3')
    if hist_unet.get(metric): ax.plot(hist_unet[metric], color=UNET_COLOR, lw=2, ls='--', label='UNet Baseline')
    ax.set_title(title, fontsize=12); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v2_metric_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.4 — Loss term breakdown (stacked area)
shared_keys  = ['tversky', 'boundary', 'focal', 'lovasz']
shared_cols  = ['#7986CB', '#EF5350', '#66BB6A', '#FFA726']
dual_keys    = shared_keys + ['align', 'aux']
dual_cols    = shared_cols + ['#AB47BC', '#26C6DA']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Loss Term Breakdown -- DualEncoderSeg v3', fontsize=14, fontweight='bold')

ax = axes[0]
ax.set_title('DualEncoderSeg v2 — Stage 2 Loss Terms', fontsize=11)
n2 = len(hist2['train_loss'])
if n2 > 0:
    stacks = [smooth(hist2.get(k, [0]*n2) or [0]*n2) for k in dual_keys]
    ax.stackplot(range(1, n2+1), stacks, labels=['Tversky','Boundary','Focal','Lovász','Align','Aux'],
                 colors=dual_cols, alpha=0.82)
    for ep, label in [(20, 'lw→0.5'), (60, 'lw→0.2')]:
        if ep < n2: ax.axvline(ep, color='k', ls='--', alpha=0.4, lw=1, label=label)
ax.set_xlabel('Epoch'); ax.legend(loc='upper right', fontsize=7, ncol=2); ax.grid(True, alpha=0.2)

ax = axes[1]
ax.set_title('UNet Baseline — Loss Terms (same 4-term suite)', fontsize=11)
nu = len(hist_unet['train_loss'])
if nu > 0:
    stacks_u = [smooth(hist_unet.get(k, [0]*nu) or [0]*nu) for k in shared_keys]
    ax.stackplot(range(1, nu+1), stacks_u, labels=['Tversky','Boundary','Focal','Lovász'],
                 colors=shared_cols, alpha=0.82)
ax.set_xlabel('Epoch'); ax.legend(loc='upper right', fontsize=8); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v3_loss_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.5a — Stage 1 dual-path loss dynamics (v3 diagnostic)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('v3 Stage 1: Dual-Path Loss Dynamics', fontsize=13, fontweight='bold')

ax = axes[0]
ax.set_title('Three Loss Terms over Epochs', fontsize=11)
if hist1.get('main'):
    ax.plot(smooth(hist1['main']),   color='#4CAF50', lw=2, label='L_main  (z+skips -> recon)')
    ax.plot(smooth(hist1['z_only']), color='#FF9800', lw=2, label='L_z_only (z alone -> recon)')
    ax.plot(smooth(hist1['lat']),    color='#9C27B0', lw=2, label='L_lat   (z -> 32x32 head)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.set_title('Val Dice (full path) -- Generalization vs v2 Saturation', fontsize=11)
if hist1.get('val_dice'):
    ax.plot(hist1['val_dice'], color='#4CAF50', lw=2, label='v3 Val Dice (z+skips)')
ax.axhline(0.9999, color='red', ls=':', lw=1.5, alpha=0.7, label='v2 saturation (0.9999)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Dice'); ax.set_ylim(0, 1.05)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v3_stage1_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

# Cell 11.5b — Stage 2 alignment loss + curriculum weight
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('DualEncoderSeg v3: Stage 2 Latent Alignment Dynamics', fontsize=13, fontweight='bold')

ax = axes[0]
ax.set_title('Alignment Loss over Epochs', fontsize=11)
if hist2.get('align') and any(hist2['align']):
    ax.plot(smooth(hist2['align']), color='#AB47BC', lw=2, label='Align (MSE+Cosine)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.set_title('Alignment Weight Schedule (Curriculum)', fontsize=11)
if hist2.get('align_w'):
    ax.step(range(1, len(hist2['align_w'])+1), hist2['align_w'],
            color='#FF7043', lw=2, where='post')
ax.set_xlabel('Epoch'); ax.set_ylabel('lambda_align'); ax.set_ylim(0, 1.2)
ax.axhline(1.0, color='gray', ls=':', alpha=0.5, label='Epoch 1-20: 1.0')
ax.axhline(0.5, color='gray', ls=':', alpha=0.5, label='Epoch 21-60: 0.5')
ax.axhline(0.2, color='gray', ls=':', alpha=0.5, label='Epoch 61+: 0.2')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v3_alignment.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 11.6 — Final metric grouped bar chart
metric_labels = ['IoU', 'Dice', 'Accuracy', 'Sensitivity', 'Specificity']
keys_in_dict  = ['iou', 'dice', 'acc', 'sensitivity', 'specificity']
dual_vals = [dual_final[k] for k in keys_in_dict]
unet_vals = [unet_final[k] for k in keys_in_dict]
x = np.arange(len(metric_labels)); w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, dual_vals, w, label='DualEncoderSeg v3', color=DUAL_COLOR, alpha=0.88, edgecolor='white')
b2 = ax.bar(x + w/2, unet_vals, w, label='UNet Baseline',     color=UNET_COLOR, alpha=0.88, edgecolor='white')
for bar in [*b1, *b2]:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.003, f'{h:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_ylim(0, 1.12); ax.set_xticks(x); ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylabel('Score'); ax.set_title('Final Validation Metrics — DualEncoderSeg v3 vs UNet Baseline', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v3_metric_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.7 — Radar chart
categories = ['IoU', 'Dice', 'Accuracy', 'Sensitivity', 'Specificity']
N = len(categories)
angles = [n / float(N) * 2 * math.pi for n in range(N)] + [0]
dual_r = dual_vals + [dual_vals[0]]
unet_r = unet_vals + [unet_vals[0]]

fig, ax = plt.subplots(1, 1, figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(math.pi / 2); ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, size=11)
ax.set_ylim(0, 1); ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], size=7, color='grey')
ax.plot(angles, dual_r, color=DUAL_COLOR, lw=2); ax.fill(angles, dual_r, color=DUAL_COLOR, alpha=0.25)
ax.plot(angles, unet_r, color=UNET_COLOR, lw=2, ls='--'); ax.fill(angles, unet_r, color=UNET_COLOR, alpha=0.15)
ax.set_title('Radar: DualEncoderSeg v3 vs UNet', size=13, fontweight='bold', pad=20)
legend_patches = [mpatches.Patch(color=DUAL_COLOR, alpha=0.7, label='DualEncoderSeg v3'),
                  mpatches.Patch(color=UNET_COLOR, alpha=0.7, label='UNet Baseline')]
ax.legend(handles=legend_patches, loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v3_radar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.8 — Visual prediction comparison grid
N_SAMPLES = 4
dual_model.eval(); unet_model.eval()
s_imgs, s_masks = [], []
for imgs, masks in val_dl:
    s_imgs.append(imgs); s_masks.append(masks)
    if sum(x.shape[0] for x in s_imgs) >= N_SAMPLES: break
s_imgs  = torch.cat(s_imgs,  0)[:N_SAMPLES].float().to(CFG.device)
s_masks = torch.cat(s_masks, 0)[:N_SAMPLES].float().to(CFG.device)

with torch.no_grad():
    dual_preds = torch.sigmoid(dual_model(s_imgs)).cpu().squeeze(1).numpy()
    unet_preds = torch.sigmoid(unet_model(s_imgs)).cpu().squeeze(1).numpy()
imgs_np  = s_imgs.cpu().permute(0,2,3,1).numpy()
masks_np = s_masks.cpu().squeeze(1).numpy()

fig, axes = plt.subplots(4, N_SAMPLES, figsize=(4*N_SAMPLES, 14))
fig.suptitle('Visual Comparison — DualEncoderSeg v3 vs UNet Baseline', fontsize=14, fontweight='bold', y=1.02)
row_labels = ['Input Image', 'Ground Truth', 'DualEncoderSeg v2', 'UNet Baseline']
data_rows  = [imgs_np, masks_np, dual_preds > 0.5, unet_preds > 0.5]
cmaps      = ['viridis', 'gray', 'Blues', 'Oranges']
bcolors    = [None, 'black', DUAL_COLOR, UNET_COLOR]

for r, (data, rlabel, cmap, bc) in enumerate(zip(data_rows, row_labels, cmaps, bcolors)):
    for c in range(N_SAMPLES):
        ax = axes[r, c]
        ax.imshow(data[c]) if data[c].ndim == 3 else ax.imshow(data[c], cmap=cmap, vmin=0, vmax=1)
        ax.axis('off')
        if c == 0: ax.set_ylabel(rlabel, fontsize=10, fontweight='bold', rotation=90, labelpad=8)
        if bc:
            for spine in ax.spines.values():
                spine.set_edgecolor(bc); spine.set_linewidth(2); spine.set_visible(True)
plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v3_visual_grid.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.9 — Threshold sweep
thresholds = np.linspace(0.1, 0.9, 17)

def threshold_sweep(model, val_loader, device, thresholds):
    model.eval()
    all_logits, all_masks = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            all_logits.append(model(imgs.float().to(device)).cpu())
            all_masks.append(masks.cpu())
    all_logits = torch.cat(all_logits); all_masks = torch.cat(all_masks)
    return [compute_metrics(all_logits, all_masks, float(t))['dice'] for t in thresholds]

print('Running threshold sweep...')
dual_sweep = threshold_sweep(dual_model, val_dl, CFG.device, thresholds)
unet_sweep = threshold_sweep(unet_model, val_dl, CFG.device, thresholds)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, dual_sweep, 'o-', color=DUAL_COLOR, lw=2, ms=5, label='DualEncoderSeg v3')
ax.plot(thresholds, unet_sweep, 's--', color=UNET_COLOR, lw=2, ms=5, label='UNet Baseline')
ax.axvline(0.5, color='gray', ls=':', alpha=0.6, label='Default (0.5)')
ax.set_xlabel('Decision Threshold', fontsize=11); ax.set_ylabel('Dice', fontsize=11)
ax.set_title('Dice vs Decision Threshold', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
best_d = thresholds[np.argmax(dual_sweep)]; best_u = thresholds[np.argmax(unet_sweep)]
print(f'Best threshold — DualEnc: {best_d:.2f} (Dice={max(dual_sweep):.4f})  |  UNet: {best_u:.2f} (Dice={max(unet_sweep):.4f})')
plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/v3_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## §12 — Inference Utilities

In [ ]:
# Cell 12.1 — predict_single / predict_batch

@torch.no_grad()
def predict_single(model, image_tensor, threshold=0.5, device=None):
    """C×H×W → H×W uint8 binary mask. Works for DualEncoderSegV2 and UNetBaseline."""
    if device is None: device = CFG.device
    model.eval()
    logits = model(image_tensor.unsqueeze(0).float().to(device))
    return (torch.sigmoid(logits) > threshold).squeeze().cpu().to(torch.uint8)


@torch.no_grad()
def predict_batch(model, images, threshold=0.5, device=None):
    """B×C×H×W → B×H×W uint8. Works for both models."""
    if device is None: device = CFG.device
    model.eval()
    logits = model(images.float().to(device))
    return (torch.sigmoid(logits) > threshold).squeeze(1).cpu().to(torch.uint8)


print('Inference utilities: predict_single, predict_batch')
print('Usage:')
print('  mask = predict_single(dual_model, image_tensor)  # DualEncoderSeg v2')
print('  mask = predict_single(unet_model, image_tensor)  # UNet Baseline')

In [ ]:
# Cell 12.2 — Summary of all saved plots and files
save_dir = Path(CFG.save_dir)
plots = sorted(save_dir.glob('v3_*.png'))
print(f'All outputs saved to: {save_dir.resolve()}')
print()
for p in plots: print(f'  {p.name}')
print()
print('Plots generated:')
print('  v3_loss_curves.png       -- Stage 1 (3-term) / Stage 2 / UNet loss curves')
print('  v3_stage1_dynamics.png   -- v3 Stage 1 dual-path loss dynamics (diagnostic)')
print('  v3_metric_curves.png     -- Val Dice & IoU over epochs')
print('  v3_loss_breakdown.png    -- Stacked area: all Stage 2 loss terms per epoch')
print('  v3_alignment.png         -- DualEnc latent alignment loss + curriculum weight')
print('  v3_metric_bars.png       -- Grouped bar chart (5 metrics)')
print('  v3_radar.png             -- Radar spider chart')
print('  v3_visual_grid.png       -- Side-by-side prediction grid')
print('  v3_threshold_sweep.png   -- Dice vs threshold')
